In [3]:
import sys
!{sys.executable} -m pip install pandas numpy scikit-learn flask joblib gunicorn

Defaulting to user installation because normal site-packages is not writeable


In [4]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

C:\Users\chman\AppData\Roaming\Python\Python311\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [5]:
df = pd.read_csv(r'C:\Users\chman\OneDrive\Desktop\music.csv')
df.head()

,age,gender,genre
0,20,1,HipHop
1,23,1,HipHop
2,25,1,HipHop
3,26,1,Jazz
4,29,1,Jazz


In [6]:
df = pd.read_csv(r'C:\Users\chman\OneDrive\Desktop\music.csv')
df.head()

,age,gender,genre
0,20,1,HipHop
1,23,1,HipHop
2,25,1,HipHop
3,26,1,Jazz
4,29,1,Jazz


In [7]:
print(df.columns)
print(df.shape)
df.head()

Index(['age', 'gender', 'genre'], dtype='object')
(18, 3)


,age,gender,genre
0,20,1,HipHop
1,23,1,HipHop
2,25,1,HipHop
3,26,1,Jazz
4,29,1,Jazz


In [8]:
df.isnull().sum()

age       0
gender    0
genre     0
dtype: int64

In [9]:
df = df.dropna()
df.isnull().sum()

age       0
gender    0
genre     0
dtype: int64

In [10]:
X = df[["age", "gender"]]
y = df["genre"]

In [11]:
gender_encoder = LabelEncoder()
df["gender"] = gender_encoder.fit_transform(df["gender"])
X = df[["age", "gender"]]

In [12]:
genre_encoder = LabelEncoder()
y_encoded = genre_encoder.fit_transform(y)

print(genre_encoder.classes_)

['Acoustic' 'Classical' 'Dance' 'HipHop' 'Jazz']


In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (14, 2)
Testing data: (4, 2)


In [14]:
model = DecisionTreeClassifier(random_state=42)

model.fit(X_train, y_train)

print("Model training completed")

Model training completed


In [15]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

print(classification_report(
    y_test,
    y_pred,
    labels=range(len(genre_encoder.classes_)),
    target_names=genre_encoder.classes_,
    zero_division=0
))

Accuracy: 1.0
              precision    recall  f1-score   support

    Acoustic       0.00      0.00      0.00         0
   Classical       1.00      1.00      1.00         1
       Dance       0.00      0.00      0.00         0
      HipHop       1.00      1.00      1.00         2
        Jazz       1.00      1.00      1.00         1

    accuracy                           1.00         4
   macro avg       0.60      0.60      0.60         4
weighted avg       1.00      1.00      1.00         4



In [16]:
sample = pd.DataFrame([[21, 1]], columns=["age", "gender"])

prediction = model.predict(sample)
predicted_genre = genre_encoder.inverse_transform(prediction)

print("Recommended Music Genre:", predicted_genre[0])

Recommended Music Genre: HipHop


In [17]:
joblib.dump(model, "music_model.pkl")
joblib.dump(genre_encoder, "genre_encoder.pkl")

print("Model saved as music_model.pkl")
print("Genre encoder saved as genre_encoder.pkl")

Model saved as music_model.pkl
Genre encoder saved as genre_encoder.pkl


In [18]:
joblib.dump(gender_encoder, "gender_encoder.pkl")
print("Gender encoder saved as gender_encoder.pkl")

Gender encoder saved as gender_encoder.pkl


In [19]:
app_code = """
from flask import Flask, render_template, request
import pandas as pd
import joblib

app = Flask(__name__)

model = joblib.load("music_model.pkl")
genre_encoder = joblib.load("genre_encoder.pkl")

@app.route("/")
def home():
    return render_template("index.html")

@app.route("/predict", methods=["POST"])
def predict():
    try:
        age = int(request.form["age"])
        gender = int(request.form["gender"])

        if age <= 0:
            return render_template(
                "index.html",
                prediction_text="Error: Age must be a positive value."
            )

        input_data = pd.DataFrame(
            [[age, gender]],
            columns=["age", "gender"]
        )

        prediction = model.predict(input_data)
        predicted_genre = genre_encoder.inverse_transform(prediction)[0]

        return render_template(
            "index.html",
            prediction_text="Recommended Music Genre: " + predicted_genre
        )

    except Exception:
        return render_template(
            "index.html",
            prediction_text="Error: Please enter valid input values."
        )

if __name__ == "__main__":
    app.run(debug=False)
"""

with open("app.py", "w") as file:
    file.write(app_code)

print("app.py created successfully")

app.py created successfully


In [20]:
os.makedirs("templates", exist_ok=True)

html_code = """
<!DOCTYPE html>
<html>
<head>
    <title>Music Recommendation App</title>

    <style>
        body {
            font-family: Arial;
            background-color: #f4f4f4;
            text-align: center;
        }

        .container {
            background: white;
            width: 420px;
            margin: 70px auto;
            padding: 30px;
            border-radius: 10px;
            box-shadow: 0px 0px 10px gray;
        }

        input, select {
            width: 90%;
            padding: 10px;
            margin: 10px;
        }

        button {
            padding: 10px;
            width: 95%;
            background: #2874a6;
            color: white;
            border: none;
            font-size: 16px;
        }

        .result {
            margin-top: 20px;
            color: green;
            font-weight: bold;
        }
    </style>
</head>

<body>

<div class="container">
    <h2>Music Genre Recommendation App</h2>

    <form action="/predict" method="post">
        <input type="number" name="age" placeholder="Enter Age" required>

        <select name="gender" required>
            <option value="">Select Gender</option>
            <option value="1">Male</option>
            <option value="0">Female</option>
        </select>

        <button type="submit">Recommend Genre</button>
    </form>

    <div class="result">
        {{ prediction_text }}
    </div>
</div>

</body>
</html>
"""

with open("templates/index.html", "w") as file:
    file.write(html_code)

print("templates/index.html created successfully")

templates/index.html created successfully


In [21]:
requirements = """
flask
pandas
numpy
scikit-learn
joblib
gunicorn
"""

with open("requirements.txt", "w") as file:
    file.write(requirements)

print("requirements.txt created successfully")

requirements.txt created successfully


In [30]:
!python app.py

^C


In [29]:
http://127.0.0.1:5000

SyntaxError: invalid syntax (2511739473.py, line 1)